# S2 — Importing and exporting data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enriquea/ZebraQ/blob/main/lessons-py/S2_Import_Export_Data.ipynb)

**ZebraQ — Introduction to Python and bulk RNA-seq data analysis**

By the end of this notebook you will be able to read a data file into pandas
whatever its format, diagnose the three things that most often go wrong
(separators, dtypes, missing values), and write your results back out.

> **Note for people coming from the R course.** The R lesson compares
> `read.csv`, `read.table` and `read.delim`. pandas has a **single** reader,
> `pd.read_csv`, which handles all of those cases through its arguments. So this
> lesson spends its time on the arguments that actually matter instead.

## 0. Setup

In [1]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install", "pandas", "openpyxl", "pyarrow"],
        check=True,
    )

from pathlib import Path
import numpy as np
import pandas as pd

# Resolve data files locally (when you cloned the repo) or from GitHub (Colab).
RAW = "https://raw.githubusercontent.com/enriquea/ZebraQ/main/data/"


def data_path(fname):
    """Return a local path if the repo is checked out, otherwise a raw GitHub URL."""
    local = Path("../data") / fname
    return local if local.exists() else RAW + fname


# Somewhere to write our outputs
OUT = Path("../results/py")
OUT.mkdir(parents=True, exist_ok=True)

print(f"pandas {pd.__version__}")
print(f"reading data from: {data_path('chd_genes.annotations.tsv')}")

pandas 2.3.3
reading data from: ../data/chd_genes.annotations.tsv


## 1. Reading delimited text

`pd.read_csv` reads any delimited text file. The **separator** is the one
argument you must get right.

In [2]:
# A comma-separated file. sep="," is the default, so it can be omitted.
df_csv = pd.read_csv(data_path("chd_genes.annotations.csv"))
print(f"CSV : {df_csv.shape[0]} rows x {df_csv.shape[1]} columns")

# A tab-separated file. This is where the R course would reach for read.delim.
df_tsv = pd.read_csv(data_path("chd_genes.annotations.tsv"), sep="\t")
print(f"TSV : {df_tsv.shape[0]} rows x {df_tsv.shape[1]} columns")

print(f"\nsame data either way: {df_csv.equals(df_tsv)}")
df_tsv.head()

CSV : 276 rows x 10 columns
TSV : 276 rows x 10 columns

same data either way: True


,gene_symbol,category,pLI,plof,gene_length,obs_lof,obs_syn,exp_lof,exp_syn,chromosome
0,ABCC9,syndromic,9.352400e-09,0.482,144002,30,298,84.399,298.160,12
1,ABL1,syndromic,9.999800e-01,0.176,173730,3,325,44.108,314.370,9
2,ACAD9,syndromic,4.525600e-08,0.814,36472,17,147,31.321,144.410,3
3,ACTA2,nonsyndromic,9.301700e-01,0.364,56317,2,72,17.293,84.674,10
4,ACTB,syndromic,9.856400e-01,0.232,36634,0,190,12.858,96.859,7


> **The R translation table**
>
> | R | Python |
> |---|---|
> | `read.csv(f)` | `pd.read_csv(f)` |
> | `read.delim(f)` | `pd.read_csv(f, sep="\t")` |
> | `read.table(f, sep="\t", header=TRUE)` | `pd.read_csv(f, sep="\t")` |
> | `read.table(f, header=FALSE)` | `pd.read_csv(f, header=None)` |
>
> One function, different arguments.

### 1.1 What if you do not know the separator?

Pass `sep=None` with `engine="python"` and pandas will work it out. Handy once,
but always write the real separator into your final script — guessing is slower
and can be wrong.

In [3]:
sniffed = pd.read_csv(data_path("chd_genes.annotations.tsv"), sep=None, engine="python")
print(f"auto-detected the separator: {sniffed.shape[0]} rows x {sniffed.shape[1]} columns")

auto-detected the separator: 276 rows x 10 columns


### 1.2 Choosing the index

`index_col` makes one column the row label — the equivalent of R's
`row.names(df) <- df$gene_symbol`.

In [4]:
indexed = pd.read_csv(
    data_path("chd_genes.annotations.tsv"), sep="\t", index_col="gene_symbol"
)
print(indexed.head(3))
print(f"\nnow you can look a gene up directly:\n{indexed.loc['ABCC9']}")

              category           pLI   plof  gene_length  obs_lof  obs_syn  \
gene_symbol                                                                  
ABCC9        syndromic  9.352400e-09  0.482       144002       30      298   
ABL1         syndromic  9.999800e-01  0.176       173730        3      325   
ACAD9        syndromic  4.525600e-08  0.814        36472       17      147   

             exp_lof  exp_syn chromosome  
gene_symbol                               
ABCC9         84.399   298.16         12  
ABL1          44.108   314.37          9  
ACAD9         31.321   144.41          3  

now you can look a gene up directly:
category       syndromic
pLI                  0.0
plof               0.482
gene_length       144002
obs_lof               30
obs_syn              298
exp_lof           84.399
exp_syn           298.16
chromosome            12
Name: ABCC9, dtype: object


## 2. Data types (dtypes)

This is the number one source of confusion for beginners. pandas guesses a type
for every column. When the guess is wrong, everything downstream misbehaves.

In [5]:
print("what pandas inferred:")
print(df_tsv.dtypes)

what pandas inferred:
gene_symbol     object
category        object
pLI            float64
plof           float64
gene_length      int64
obs_lof          int64
obs_syn          int64
exp_lof        float64
exp_syn        float64
chromosome      object
dtype: object


`object` almost always means **text**. Notice `chromosome` is `object`, not a
number — because it contains `X` and `Y` alongside the digits. That is correct
here, but it means you cannot do arithmetic on it.

In [6]:
print(f"unique chromosome values: {sorted(df_tsv['chromosome'].unique(), key=str)}")

unique chromosome values: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '3', '4', '5', '6', '7', '8', '9', 'X']


### 2.1 Forcing a type

Use `dtype=` when reading, or `.astype()` afterwards.

In [7]:
# Force chromosome to be text explicitly, so "1" never becomes the number 1
typed = pd.read_csv(
    data_path("chd_genes.annotations.tsv"), sep="\t", dtype={"chromosome": str}
)
print(f"chromosome dtype: {typed['chromosome'].dtype}")

# Convert a column to a different numeric type
print(f"gene_length as float: {df_tsv['gene_length'].astype(float).head(3).tolist()}")

chromosome dtype: object
gene_length as float: [144002.0, 173730.0, 36472.0]


### 2.2 When a column will not convert

`pd.to_numeric` with `errors="coerce"` turns anything unparseable into `NaN`
instead of raising. This is the standard way to clean a messy numeric column.

In [8]:
messy = pd.Series(["1.5", "2.7", "not_measured", "4.1", ""])
print("original (all text):")
print(messy.tolist())

cleaned = pd.to_numeric(messy, errors="coerce")
print("\nafter pd.to_numeric(errors='coerce'):")
print(cleaned.tolist())
print(f"\ndtype is now {cleaned.dtype}, with {cleaned.isna().sum()} missing values")

original (all text):
['1.5', '2.7', 'not_measured', '4.1', '']

after pd.to_numeric(errors='coerce'):
[1.5, 2.7, nan, 4.1, nan]

dtype is now float64, with 2 missing values


## 3. Missing values

R has a single `NA`. Python has several things that mean "missing", and they
behave differently.

In [9]:
print(f"np.nan is a float          : {type(np.nan)}")
print(f"None is Python's null      : {type(None)}")
print(f"pd.NA is pandas' own null  : {type(pd.NA)}")

print("\nThe rule that catches everyone:")
print(f"  np.nan == np.nan  ->  {np.nan == np.nan}   (!!) never compare with ==")
print(f"  pd.isna(np.nan)   ->  {pd.isna(np.nan)}   use pd.isna() instead")

np.nan is a float          : <class 'float'>
None is Python's null      : <class 'NoneType'>
pd.NA is pandas' own null  : <class 'pandas._libs.missing.NAType'>

The rule that catches everyone:
  np.nan == np.nan  ->  False   (!!) never compare with ==
  pd.isna(np.nan)   ->  True   use pd.isna() instead


### 3.1 Finding missing values in real data

In [10]:
na_counts = df_tsv.isna().sum()
print("missing values per column:")
print(na_counts)
print(f"\ntotal missing in the whole table: {df_tsv.isna().sum().sum()}")

missing values per column:
gene_symbol    0
category       0
pLI            0
plof           0
gene_length    0
obs_lof        0
obs_syn        0
exp_lof        0
exp_syn        0
chromosome     0
dtype: int64

total missing in the whole table: 0


### 3.2 Telling pandas what counts as missing

Real files encode missing data in creative ways. `na_values=` handles them.

In [11]:
demo = pd.DataFrame({"gene": ["A", "B", "C", "D"], "score": ["1.0", "NA", "-999", "3.5"]})
demo_file = OUT / "_demo_missing.csv"
demo.to_csv(demo_file, index=False)

naive = pd.read_csv(demo_file)
print(f"read naively   -> dtype {naive['score'].dtype}, values {naive['score'].tolist()}")

smart = pd.read_csv(demo_file, na_values=["NA", "-999"])
print(f"with na_values -> dtype {smart['score'].dtype}, values {smart['score'].tolist()}")
print("\n'-999' was a sentinel for 'no measurement'. Without na_values it would have")
print("been treated as a real, very negative number and wrecked every statistic.")

read naively   -> dtype float64, values [1.0, nan, -999.0, 3.5]
with na_values -> dtype float64, values [1.0, nan, nan, 3.5]

'-999' was a sentinel for 'no measurement'. Without na_values it would have
been treated as a real, very negative number and wrecked every statistic.


### 3.3 Handling them

In [12]:
print(f"drop rows with any NA : {len(smart.dropna())} of {len(smart)} rows kept")
print(f"fill with a value     : {smart['score'].fillna(0).tolist()}")
print(f"fill with the mean    : {smart['score'].fillna(smart['score'].mean()).round(2).tolist()}")

drop rows with any NA : 2 of 4 rows kept
fill with a value     : [1.0, 0.0, 0.0, 3.5]
fill with the mean    : [1.0, 2.25, 2.25, 3.5]


## 4. Excel files

`pd.read_excel` needs the `openpyxl` package for `.xlsx` files.

In [13]:
df_xlsx = pd.read_excel(data_path("chd_genes.annotations.xlsx"))
print(f"read from Excel: {df_xlsx.shape[0]} rows x {df_xlsx.shape[1]} columns")
df_xlsx.head(3)

read from Excel: 276 rows x 10 columns


,gene_symbol,category,pLI,plof,gene_length,obs_lof,obs_syn,exp_lof,exp_syn,chromosome
0,ABCC9,syndromic,9.352400e-09,0.482,144002,30,298,84.399,298.16,12
1,ABL1,syndromic,9.999800e-01,0.176,173730,3,325,44.108,314.37,9
2,ACAD9,syndromic,4.525600e-08,0.814,36472,17,147,31.321,144.41,3


For a workbook with several sheets, `sheet_name=None` reads them all into a
dictionary of DataFrames:

```python
all_sheets = pd.read_excel("workbook.xlsx", sheet_name=None)
all_sheets["Sheet1"]        # each key is a sheet name
```

> **In R:** `readxl::read_excel(path, sheet = "Sheet1")`.

## 5. Writing data out

> ### ⚠️ The `index=False` trap
> By default pandas writes the row index as an extra unnamed first column. Read
> that file back and you get a junk column called `Unnamed: 0`.
>
> This is exactly the same trap as R's `write.csv(iris, "iris.csv")`, which
> writes a leading `""` column of row numbers. In R you avoid it with
> `row.names = FALSE`; in pandas with `index=False`.

In [14]:
small = df_tsv.head(5)

with_index = OUT / "_with_index.csv"
without_index = OUT / "_without_index.csv"

small.to_csv(with_index)                  # the trap
small.to_csv(without_index, index=False)  # what you almost always want

back_bad = pd.read_csv(with_index)
back_good = pd.read_csv(without_index)

print(f"written WITH index, read back    -> columns: {list(back_bad.columns)[:3]} ...")
print(f"written WITHOUT index, read back -> columns: {list(back_good.columns)[:3]} ...")
print(f"\nspot the junk column: {'Unnamed: 0' in back_bad.columns}")

written WITH index, read back    -> columns: ['Unnamed: 0', 'gene_symbol', 'category'] ...
written WITHOUT index, read back -> columns: ['gene_symbol', 'category', 'pLI'] ...

spot the junk column: True


### 5.1 The other formats

In [15]:
small.to_csv(OUT / "annotations_head.tsv", sep="\t", index=False)
small.to_excel(OUT / "annotations_head.xlsx", index=False)
print(f"files now in {OUT}:")
for f in sorted(OUT.glob("*")):
    if not f.name.startswith("_"):
        print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

files now in ../results/py:
  TopGenes_perChamber.xlsx  (12,399 bytes)
  annotations_head.tsv  (387 bytes)
  annotations_head.xlsx  (5,382 bytes)
  normalised_counts.tsv  (1,611,479 bytes)
  pydeseq2_results.tsv  (1,722,691 bytes)
  pydeseq2_results.xlsx  (1,270,120 bytes)


## 6. Does the file format matter? Measure it.

The R lesson benchmarks `read.csv` against `read.table` and `read.delim`. In
pandas there is only one reader, so instead we ask a question that has a real
answer: **is a text file the best way to store a big table?**

We compare CSV against **Parquet**, a compressed binary columnar format. We use
the full gene count matrix so the numbers mean something.

`%timeit` is Python's equivalent of R's `microbenchmark`.

In [16]:
counts = pd.read_csv(data_path("salmon.merged.gene_counts.filtered.tsv"), sep="\t")
print(f"benchmarking on {counts.shape[0]:,} rows x {counts.shape[1]} columns")

csv_file = OUT / "_bench.csv"
parquet_file = OUT / "_bench.parquet"

counts.to_csv(csv_file, index=False)
counts.to_parquet(parquet_file, index=False)

print(f"\nfile size on disk")
print(f"  CSV     : {csv_file.stat().st_size:>10,} bytes")
print(f"  Parquet : {parquet_file.stat().st_size:>10,} bytes")
print(f"  Parquet is {csv_file.stat().st_size / parquet_file.stat().st_size:.1f}x smaller")

benchmarking on 14,650 rows x 7 columns



file size on disk
  CSV     :    642,928 bytes
  Parquet :    374,635 bytes
  Parquet is 1.7x smaller


> ### ⚠️ Always warm up before you benchmark
> The *first* call to a library often includes one-off costs — importing
> submodules, allocating buffers — that have nothing to do with the operation you
> are trying to measure. If you time the first call you measure the wrong thing.
>
> So we call each reader once and throw the result away, *then* start timing.

In [17]:
# Warm-up: pay the one-off import cost now, outside the measurement.
_ = pd.read_csv(csv_file)
_ = pd.read_parquet(parquet_file)
print("warm-up done — both readers are now loaded")

warm-up done — both readers are now loaded


In [18]:
print("READ speed — CSV:")
%timeit -n 5 -r 5 pd.read_csv(csv_file)

READ speed — CSV:


15.2 ms ± 5.74 ms per loop (mean ± std. dev. of 5 runs, 5 loops each)


In [19]:
print("READ speed — Parquet:")
%timeit -n 5 -r 5 pd.read_parquet(parquet_file)

READ speed — Parquet:
4.35 ms ± 515 μs per loop (mean ± std. dev. of 5 runs, 5 loops each)


Compare the two numbers above. Parquet is normally several times faster to read
and, as we saw, smaller on disk.

> If Parquet does not look faster in your run, you are probably measuring a table
> too small for the difference to show. The gap widens with size.

**The practical rule:** use CSV or TSV when a human or another program needs to
read the file. Use Parquet for intermediate results inside your own pipeline.

### 6.1 Parquet remembers your types. CSV does not.

A CSV file is only text, so every time you read one pandas has to *guess* the
type of each column. On the count matrix above the guess happens to be right,
so a CSV round trip looks lossless:

In [20]:
roundtrip_csv = pd.read_csv(csv_file)
roundtrip_parquet = pd.read_parquet(parquet_file)

print("count matrix — original :", counts.dtypes.value_counts().to_dict())
print("count matrix — via CSV  :", roundtrip_csv.dtypes.value_counts().to_dict())
print("            identical?  :", counts.equals(roundtrip_csv))

count matrix — original : {dtype('float64'): 6, dtype('O'): 1}
count matrix — via CSV  : {dtype('float64'): 6, dtype('O'): 1}
            identical?  : True


But the guess is not always right. Here is a table with three columns that are
easy to get wrong: a zero-padded identifier, a category, and a date.

In [21]:
tricky = pd.DataFrame({
    "sample_id": ["007", "013", "021"],                 # leading zeros matter
    "chamber": pd.Categorical(["LA", "RV", "LA"]),      # a category, not text
    "collected": pd.to_datetime(["2024-01-05", "2024-02-11", "2024-03-02"]),
})
print("original dtypes:")
print(tricky.dtypes)

original dtypes:
sample_id            object
chamber            category
collected    datetime64[ns]
dtype: object


In [22]:
tricky.to_csv(OUT / "_tricky.csv", index=False)
tricky.to_parquet(OUT / "_tricky.parquet", index=False)

via_csv = pd.read_csv(OUT / "_tricky.csv")
via_parquet = pd.read_parquet(OUT / "_tricky.parquet")

print("after a CSV round trip:")
print(via_csv.dtypes)
print(f"\nthe identifier became: {via_csv['sample_id'].tolist()}   <- leading zeros gone!")
print(f"identical to original? {tricky.equals(via_csv)}")

after a CSV round trip:
sample_id     int64
chamber      object
collected    object
dtype: object

the identifier became: [7, 13, 21]   <- leading zeros gone!
identical to original? False


In [23]:
print("after a Parquet round trip:")
print(via_parquet.dtypes)
print(f"\nthe identifier is still: {via_parquet['sample_id'].tolist()}")
print(f"identical to original? {tricky.equals(via_parquet)}")

after a Parquet round trip:
sample_id            object
chamber            category
collected    datetime64[ns]
dtype: object

the identifier is still: ['007', '013', '021']
identical to original? True


`sample_id` came back as an integer and lost its leading zeros, `chamber` came
back as plain text instead of a category, and `collected` came back as a string
rather than a date. Parquet stored all three correctly because the format records
the type alongside the data.

This is a real source of silent bugs in bioinformatics, where sample and gene
identifiers very often look like numbers but are not.

In [24]:
# tidy up the scratch files this notebook created
for f in OUT.glob("_*"):
    f.unlink()
print("temporary benchmark files removed")

temporary benchmark files removed


## 7. Exercises

1. Read `chd_genes.annotations.tsv` using `gene_symbol` as the index, then look
   up the row for `GATA4`.
2. How many genes have a missing `pLI` value?
3. Read the file forcing `chromosome` to be text and `gene_length` to be a float.
   Confirm with `.dtypes`.
4. Write the `syndromic` genes only to an Excel file, without the index.
5. Save the count matrix as Parquet, read it back, and confirm the dtypes
   survived the round trip (they do not survive a CSV round trip).

## 8. Session information

In [25]:
import session_info

session_info.show()